# Customer Churn — Segmentation & Rule-Based Risk Model

**Internship task:** Task 04 · Data Analytics  
**Purpose:** segment customers using behavioral/account signals and create an explainable triage score.

## Decision question
Which customer groups should a fictional subscription team contact first to investigate avoidable churn?

This is a **transparent heuristic, not a trained predictive model**. Score points and thresholds are policy choices, not statistically fitted weights or churn probabilities. The supplied dataset has only 15 synthetic records. Same-sample associations are descriptive and must not be treated as out-of-sample model accuracy or causation.

## 1. Load and profile the supplied data

The CSV is included under data/ so this task folder is self-contained. Customer IDs are identifiers, not numeric measures.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "cleaned_customer_churn.csv").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "cleaned_customer_churn.csv"
OUT_PATH = ROOT / "outputs" / "customer_segment_assignments.csv"

df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df)} | columns: {df.shape[1]}")
print("Missing cells:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Duplicate CustomerID values:", int(df["CustomerID"].duplicated().sum()))
display(df.head())
display(pd.DataFrame({"dtype":df.dtypes.astype(str),"missing":df.isna().sum(),"distinct":df.nunique(dropna=True)}))

## 2. Feature selection and rationale

- **Contract type:** account commitment signal. The supplied values are Month-to-Month, One Year, and Two Year; these labels are preserved.
- **Tenure months:** completed months since signup. Recent tenure receives more points as a review heuristic.
- **Support tickets:** count in the previous 90 days. Recent support activity is an actionable service signal.

CustomerID is never scored. Gender and age are excluded. Monthly charges are not a score input; they only summarize charges associated with already-churned records. Payment method and subscription type are not given risk points because this tiny synthetic sample does not support defensible weights for them.

## 3. Define a reproducible rule-based score

| Feature | Condition | Points |
|---|---|---:|
| Contract | Month-to-Month | 35 |
| Contract | One Year | 15 |
| Contract | Two Year | 5 |
| Tenure | 0–6 months | 25 |
| Tenure | 7–12 months | 18 |
| Tenure | 13–24 months | 8 |
| Tenure | 25+ months | 0 |
| Support tickets | 5+ | 25 |
| Support tickets | 3–4 | 15 |
| Support tickets | 1–2 | 5 |
| Support tickets | 0 | 0 |

Total score = contract + tenure + support tickets. The theoretical range is **0–85**. Risk bands: Low 0–34, Medium 35–59, High 60–85. These weights and boundaries are simple starting assumptions, not a learned model or probability.

In [ ]:
CONTRACT_POINTS = {"Month-to-Month": 35, "One Year": 15, "Two Year": 5}
MAX_SCORE = 85

def tenure_points(value):
    if pd.isna(value) or not np.isfinite(float(value)) or float(value) < 0 or not float(value).is_integer():
        raise ValueError("TenureMonths must be a finite, non-negative integer.")
    value = int(value)
    if value <= 6: return 25
    if value <= 12: return 18
    if value <= 24: return 8
    return 0

def support_points(value):
    if pd.isna(value) or not np.isfinite(float(value)) or float(value) < 0 or not float(value).is_integer():
        raise ValueError("SupportTickets must be a finite, non-negative integer.")
    value = int(value)
    if value >= 5: return 25
    if value >= 3: return 15
    if value >= 1: return 5
    return 0

def risk_band_from_score(score):
    if pd.isna(score) or not np.isfinite(float(score)) or float(score) < 0 or float(score) > MAX_SCORE:
        raise ValueError("Risk score must be finite and between 0 and 85.")
    if score >= 60: return "High"
    if score >= 35: return "Medium"
    return "Low"

def calculate_score(row):
    contract = row.get("ContractType")
    if pd.isna(contract) or contract not in CONTRACT_POINTS:
        raise ValueError(f"Unknown or missing ContractType: {contract!r}")
    contract_score = CONTRACT_POINTS[contract]
    tenure_score = tenure_points(row.get("TenureMonths"))
    tickets_score = support_points(row.get("SupportTickets"))
    score = contract_score + tenure_score + tickets_score
    assert 0 <= score <= MAX_SCORE
    return {
        "ContractPoints": contract_score,
        "TenurePoints": tenure_score,
        "SupportPoints": tickets_score,
        "RiskScore": score,
        "RiskBand": risk_band_from_score(score)
    }

print("Functions defined. Maximum theoretical score:", MAX_SCORE)

## 4. Define mutually exclusive behavioral segments

Apply rules in this order so each valid customer receives exactly one segment:

1. **Early tenure + elevated support:** tenure <=12 months and >=3 tickets.
2. **Month-to-month + continuing support signal:** Month-to-Month contract, but not in segment 1.
3. **Established + low support demand:** tenure >=13 months and <=2 tickets.
4. **Mixed signals — monitor:** remaining valid customers.

These are operational triage labels, not personas proven by statistical clustering.

In [ ]:
def assign_segment(row):
    tenure = int(row["TenureMonths"])
    tickets = int(row["SupportTickets"])
    contract = row["ContractType"]
    if tenure <= 12 and tickets >= 3:
        return ("Early tenure + elevated support",
                "Proactive service check-in; review recent ticket themes, resolution time, and onboarding friction.")
    if contract == "Month-to-Month":
        return ("Month-to-month + continuing support signal",
                "Offer a helpful pre-renewal check-in; understand unresolved needs before proposing a plan change.")
    if tenure >= 13 and tickets <= 2:
        return ("Established + low support demand",
                "Use light-touch relationship maintenance; monitor without unnecessary contact.")
    return ("Mixed signals — monitor",
            "Review account context manually before intervention; do not auto-contact from score alone.")

scored_rows = []
for _, row in df.iterrows():
    result = calculate_score(row)
    segment, action = assign_segment(row)
    result["BehavioralSegment"] = segment
    result["ScoreBreakdown"] = f"Contract {result['ContractPoints']} + tenure {result['TenurePoints']} + support tickets {result['SupportPoints']} = {result['RiskScore']}/85"
    result["RecommendedAction"] = action
    scored_rows.append(result)

assignments = pd.concat([df.reset_index(drop=True), pd.DataFrame(scored_rows)], axis=1)
assignments["ModelWarning"] = "Heuristic triage score, not a churn probability; validate independently."
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
assignments.to_csv(OUT_PATH, index=False)
print("Scored rows:", len(assignments))
print("Observed score range:", int(assignments.RiskScore.min()), "to", int(assignments.RiskScore.max()))
print("Export:", OUT_PATH)
display(assignments[["CustomerID","ContractType","TenureMonths","SupportTickets","ContractPoints","TenurePoints","SupportPoints","RiskScore","RiskBand","BehavioralSegment"]])

## 5. Validate edge cases

The score should never silently convert invalid or missing inputs into Low risk. The following tests validate feature boundaries, risk-band thresholds, the theoretical score bounds, unique IDs, and invalid inputs.

In [ ]:
# Tenure boundaries
assert [tenure_points(x) for x in [0, 6, 7, 12, 13, 24, 25]] == [25, 25, 18, 18, 8, 8, 0]
# Ticket boundaries
assert [support_points(x) for x in [0, 1, 2, 3, 4, 5]] == [0, 5, 5, 15, 15, 25]
# Risk-band cutoffs
assert [risk_band_from_score(x) for x in [0, 34, 35, 59, 60, 85]] == ["Low", "Low", "Medium", "Medium", "High", "High"]
assert calculate_score({"ContractType":"Month-to-Month","TenureMonths":0,"SupportTickets":5})["RiskScore"] == 85

invalid_cases = [
    {"ContractType":None,"TenureMonths":2,"SupportTickets":1},
    {"ContractType":"Unknown","TenureMonths":2,"SupportTickets":1},
    {"ContractType":"One Year","TenureMonths":-1,"SupportTickets":1},
    {"ContractType":"One Year","TenureMonths":2.5,"SupportTickets":1},
    {"ContractType":"One Year","TenureMonths":2,"SupportTickets":-1},
    {"ContractType":"One Year","TenureMonths":2,"SupportTickets":np.nan},
]
for case in invalid_cases:
    try: calculate_score(case)
    except (ValueError, TypeError): pass
    else: raise AssertionError(f"Invalid case was not rejected: {case}")

for value in [-1, 86, np.nan]:
    try: risk_band_from_score(value)
    except ValueError: pass
    else: raise AssertionError(f"Invalid risk score accepted: {value}")

assert assignments.CustomerID.is_unique
assert assignments.RiskScore.between(0, MAX_SCORE).all()
assert assignments.RiskBand.isin(["Low","Medium","High"]).all()
assert assignments.BehavioralSegment.notna().all()
print("PASS: feature boundaries, risk thresholds, maximum score, unique IDs, and 9 invalid-edge checks.")

## 6. Review risk bands and segments

Churn outcome is used only for descriptive review after scoring. The score was not fitted on this dataset, and the observed rates below are not an independent test of predictive performance.

In [ ]:
N = len(assignments)
churned_count = int((assignments["Churn"] == "Yes").sum())
print(f"Overall sample churn rate: {churned_count}/{N} = {churned_count/N:.1%}")

risk_summary = assignments.groupby("RiskBand").agg(
    Customers=("CustomerID","count"),
    Churned=("Churn",lambda s:int((s=="Yes").sum())),
    AverageTenureMonths=("TenureMonths","mean"),
    AverageSupportTickets=("SupportTickets","mean")
).reindex(["High","Medium","Low"])
risk_summary["ObservedChurnRate"] = risk_summary["Churned"] / risk_summary["Customers"]
risk_summary["MonthlyChargesAtRisk"] = [
    assignments.loc[(assignments.RiskBand==band)&(assignments.Churn=="Yes"),"MonthlyCharges"].sum()
    for band in risk_summary.index
]
display(risk_summary.round(3))

segment_summary = assignments.groupby("BehavioralSegment").agg(
    Customers=("CustomerID","count"),
    Churned=("Churn",lambda s:int((s=="Yes").sum())),
    AverageTenureMonths=("TenureMonths","mean"),
    AverageSupportTickets=("SupportTickets","mean")
)
segment_summary["ObservedChurnRate"] = segment_summary["Churned"] / segment_summary["Customers"]
segment_summary["MonthlyChargesAtRisk"] = [
    assignments.loc[(assignments.BehavioralSegment==segment)&(assignments.Churn=="Yes"),"MonthlyCharges"].sum()
    for segment in segment_summary.index
]
display(segment_summary.sort_values("Customers",ascending=False).round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
assignments["RiskBand"].value_counts().reindex(["High","Medium","Low"],fill_value=0).plot(kind="bar",ax=axes[0],title="Customers by rule-based risk band")
axes[0].set_xlabel("Risk band"); axes[0].set_ylabel("Customer count")
(assignments.groupby("RiskBand")["Churn"].apply(lambda s:(s=="Yes").mean()*100).reindex(["High","Medium","Low"])
 ).plot(kind="bar",ax=axes[1],title="Observed churn by risk band")
axes[1].set_xlabel("Risk band"); axes[1].set_ylabel("Churn rate in this sample (%)")
segment_summary["Customers"].sort_values(ascending=False).plot(kind="bar",ax=axes[2],title="Customers by behavioral segment")
axes[2].set_xlabel("Behavioral segment"); axes[2].set_ylabel("Customer count")
axes[2].tick_params(axis="x",labelrotation=70,labelsize=8)
plt.tight_layout(); plt.show()

## 7. Recommendation and limitations

### Recommended priority
Start with **Early tenure + elevated support**: tenure <=12 months and >=3 tickets in the prior 90 days. Review ticket themes and resolution status before contacting each customer.

### Proposed measurement
Run a time-limited, human-reviewed pilot; define a comparable baseline and follow-up window first. Compare subsequent churn and ticket-resolution outcomes for contacted versus comparable non-contacted accounts.

### Limitations
- Only 15 synthetic customers; subgroup rates are highly unstable.
- Weights are human-set assumptions, not trained coefficients.
- Same-sample rates do not demonstrate generalization or causality.
- Ticket topics, satisfaction, outreach history, renewal date, and intervention outcomes are unavailable.
- Low score is not a guarantee of retention; incomplete data should be routed for manual review.